<a href="https://colab.research.google.com/github/salmangeelani/salmangeelani/blob/main/Using%20GDELT%20/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install newspaper3k warcio yfinance tqdm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 58.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.6 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=6626ec1544feb5f4ecbac6b2a10c998015b2ab5cb82fdebbe692fcf898b53012
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=d3253f26c59295d338abc90784be10de67b146d30bc2bf8382f11794a8b749bb
  Stored in 

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger') # Often needed for newspaper3k parsing

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
!pip install lxml_html_clean

In [5]:
# Loading libraries
import gc
import gzip
import io
import nltk
import pandas as pd
import plotly.graph_objects as go
import re
import requests
import time
import tqdm
import warcio
import yfinance as yf
import zipfile

from datetime import datetime, timedelta
from newspaper import Article
from nltk.corpus import stopwords
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
#URL for GDELT GKG data
url = "http://data.gdeltproject.org/gdeltv2/20240930081500.gkg.csv.zip"

#Download and save the zipped file
response = requests.get(url, stream = True)
with open("gdelt_data.csv.zip", "wb") as file:
  for chunk in response.iter_content(chunk_size = 1024):
    if chunk:
      file.write(chunk)

#unzip the file
!unzip gdelt_data.csv.zip -d gdelt_data

#Read the SCV file into a pandas DataFrame
file_path = "gdelt_data/20240930081500.gkg.csv"
df = pd.read_csv(file_path, sep='\t', header=None)

# Add column names ( replace with actual column names from GDELT documentation)
gkg_columns = ["GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName", "DocumentIdentifier", "Counts", "V2Counts", "Themes", "V2Themes",
    "Locations", "V2Locations", "Persons", "V2Persons", "Organizations",
    "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage",
    "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations",
    "AllNames", "Amounts", "TranslationInfo", "Extras"
]
df.columns = gkg_columns

# Filter for news related netflix(adjust column and keyword as needed)
netflix_news = df[df['Organizations'].str.contains("netflix", na=False)].copy()
netflix_news

Archive:  gdelt_data.csv.zip
replace gdelt_data/20240930081500.gkg.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: gdelt_data/20240930081500.gkg.csv  


,GKGRECORDID,DATE,SourceCollectionIdentifier,SourceCommonName,DocumentIdentifier,Counts,V2Counts,Themes,V2Themes,Locations,...,GCAM,SharingImage,RelatedImages,SocialImageEmbeds,SocialVideoEmbeds,Quotations,AllNames,Amounts,TranslationInfo,Extras
3,20240930081500-3,20240930081500,1,malverngazette.co.uk,https://www.malverngazette.co.uk/news/24617666...,NaN,NaN,UNREST_CRACKDOWN;TAX_ECON_PRICE;USPEC_POLICY1;...,"USPEC_POLICY1,1700;EPU_POLICY_POLICY,1700;UNGP...",NaN,...,"wc:304,c1.3:1,c12.1:26,c12.10:20,c12.12:10,c12...",https://www.malverngazette.co.uk/resources/ima...,NaN,NaN,NaN,NaN,"Martin Lewis,559;Money Saving Expert,763;Extra...","99,a month if they,183;99,for either,1217;",NaN,<PAGE_LINKS>https://t.co/KOW1ZZxoF3;https://tw...
208,20240930081500-208,20240930081500,1,campaignasia.com,https://www.campaignasia.com:443/article/how-c...,NaN,NaN,EPU_ECONOMY_HISTORIC;TAX_FNCACT;TAX_FNCACT_FOU...,"EPU_ECONOMY_HISTORIC,267;EPU_ECONOMY_HISTORIC,...","1#China#CH#CH#35#105#CH;4#To Cheng, Guangdong,...",...,"wc:573,c12.1:45,c12.10:78,c12.12:16,c12.13:24,...",https://cdn.i.haymarketmedia.asia/?n=campaign-...,NaN,NaN,https://youtube.com/user/CampaignAsia;,NaN,"New Marketing Paradigm',103;Charlene Ree,338;F...","70,different languages,1398;5,different labels...",NaN,<PAGE_LINKS>https://www.iabhongkong.com/joinC2...
280,20240930081500-280,20240930081500,1,tomsguide.com,https://www.tomsguide.com/entertainment/netfli...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_GUIDE;,"TAX_FNCACT_GUIDE,814;TAX_FNCACT_GUIDE,1805;TAX...",1#United States#US#US#39.828175#-98.5795#US;1#...,...,"wc:1455,c1.3:2,c12.1:128,c12.10:167,c12.12:63,...",https://cdn.mos.cms.futurecdn.net/5hRcycNRCD2d...,https://img.youtube.com/vi/UKFMYWNatQM/maxresd...,NaN,https://youtube.com/vi/PRZ1ELeGepo/maxresdefau...,1477|34||Tim Dillon : This is Your Country#234...,"Tim Dillon,994;Official Trailer,1074;Official ...","12,singles for a dating,165;7,premiere,2300;7,...",NaN,<PAGE_LINKS>https://www.anrdoezrs.net/links/89...
530,20240930081500-530,20240930081500,1,thenorthernecho.co.uk,https://www.thenorthernecho.co.uk/news/2461766...,NaN,NaN,UNREST_CRACKDOWN;TAX_ECON_PRICE;USPEC_POLICY1;...,"UNGP_FORESTS_RIVERS_OCEANS,1721;USPEC_POLICY1,...",NaN,...,"wc:304,c1.3:1,c12.1:26,c12.10:20,c12.12:10,c12...",https://www.thenorthernecho.co.uk/resources/im...,NaN,NaN,NaN,NaN,"Martin Lewis,559;Money Saving Expert,763;Extra...","99,a month if they,183;99,for either,1217;",NaN,<PAGE_LINKS>https://t.co/KOW1ZZxoF3;https://tw...
552,20240930081500-552,20240930081500,1,tomsguide.com,https://www.tomsguide.com/entertainment/stream...,NaN,NaN,MANMADE_DISASTER_IMPLIED;MEDIA_MSM;WB_678_DIGI...,"ALLIANCE,1766;MANMADE_DISASTER_IMPLIED,44;MEDI...","3#Hollywood, California, United States#US#USCA...",...,"wc:654,c12.1:74,c12.10:71,c12.12:20,c12.13:19,...",https://cdn.mos.cms.futurecdn.net/PYKEwCu69pnq...,https://img.youtube.com/vi/3HraByA23eA/maxresd...,NaN,https://youtube.com/vi/FRoEjAlhmsw/maxresdefau...,NaN,"Love Is Blind',550;Blind Season,597;Official T...","7,| Official Trailer |,464;7,| Official Traile...",NaN,<PAGE_LINKS>https://www.tomsguide.com/entertai...
943,20240930081500-943,20240930081500,1,thenorthernecho.co.uk,https://www.thenorthernecho.co.uk/news/2461762...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_SECRETARY;WB_696_PUBLIC_...,"SOC_GENERALCRIME,388;EPU_CATS_MIGRATION_FEAR_F...",1#United Kingdom#UK#UK#54#-4#UK,...,"wc:264,c12.1:19,c12.10:29,c12.12:11,c12.13:10,...",https://www.thenorthernecho.co.uk/resources/im...,NaN,NaN,NaN,NaN,"Culture Secretary Lisa Nandy,30;Justice Secret...",NaN,NaN,<PAGE_LINKS>https://www.mirror.co.uk/news/poli...


In [11]:
# Split V2Tone into separate columns
netflix_news[['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Activity', 'Self Direction', 'WordCount']] = netflix_news['V2Tone'].str.split(',', expand=True)

#Convert numerice columns tp appropriate data types
numeric_columns = ['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Activity', 'Self Direction']
netflix_news[numeric_columns] = netflix_news[numeric_columns].astype(float).round(3)
netflix_news[['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Activity', 'Self Direction', 'WordCount']]

,Tone,Positive Score,Negative Score,Polarity,Activity,Self Direction,WordCount
3,-2.102,0.601,2.703,3.303,28.228,3.303,304
208,3.086,3.858,0.772,4.630,22.685,0.463,573
280,-0.493,4.310,4.803,9.113,20.135,1.293,1455
530,-2.102,0.601,2.703,3.303,28.228,3.303,304
552,0.682,4.502,3.820,8.322,23.602,0.955,654
943,-3.460,2.076,5.536,7.612,26.644,1.730,264


**Method 1: Averaging Tone for Each 15-Minute File and Comparing Average Scores**

In [18]:
# Generate timestamps from 8 AM to 11:45 PM on September 23, 2024
start_time = datetime(2024, 9, 23, 8, 0, 0)
end_time = datetime(2024, 9, 23, 23, 45, 0)
timestamps = []
current_time = start_time
while current_time <= end_time:
  timestamps.append(current_time.strftime("%Y%m%d%H%M%S")) # Corrected to use 'timestamps'
  current_time += timedelta(minutes=15)

# Create a dictionary to store average tone for each timestamp
avg_tones = {}

# Loop through timestamps, download, process, and discard each file
for current_timestamp_str in timestamps: # Renamed loop variable to avoid clash
  url = f"http://data.gdeltproject.org/gdeltv2/{current_timestamp_str}.gkg.csv.zip"
  response = requests.get(url, stream=True)

  # Process the zip file in memory without extracting to disk
  # This block must be inside the timestamp loop
  with zipfile.ZipFile(io.BytesIO(response.content), 'r') as zip_ref:
    for file_name in zip_ref.namelist():
      if file_name.endswith(".gkg.csv"):
        with zip_ref.open(file_name) as file: # Corrected zip.ref to zip_ref
          # Specify the encoding as 'latin-1' to handle potential encoding issues
          df = pd.read_csv(file, sep='\t', header=None,
                           on_bad_lines='skip', # skip lines with errors
                           engine='python', # use Python engine to handle large files
                           encoding='latin-1')  #explicitly set encoding to 'latin-1'

          # Add column names (refer to GDELT documentation)
          gkg_columns = [
            "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName", "DocumentIdentifier", "Counts", "V2Counts", "Themes", "V2Themes",
            "Locations", "V2Locations", "Persons", "V2Persons", "Organizations", "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage",
            "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations", "AllNames", "Amounts", "TranslationInfo", "Extras"
            ]

          df.columns = gkg_columns

          # Filter for news related netflix
          netflix_news = df[df['Organizations'].str.contains("netflix", na=False)].copy()

          # Extract Tone Components (similar to previous code)
          netflix_news[['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Activity', 'Self Direction', 'WordCount']] = netflix_news['V2Tone'].str.split(',', expand=True)
          # Renamed for clarity and correctness, as the previous 'numeric_columns' was a smaller list
          tone_numeric_columns = ['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Activity', 'Self Direction', 'WordCount']
          netflix_news[tone_numeric_columns] = netflix_news[tone_numeric_columns].astype(float, errors='ignore').round(3)

          # Calculate and store average Tone
          current_avg_tone = netflix_news['Tone'].mean()
          avg_tones[current_timestamp_str] = current_avg_tone

# File is automatically discarded when exiting the 'with' block

#Create a DataFrame from the avg_tones dict converting 'Timestamp' column to datetime objects
tone_df = pd.DataFrame(list(avg_tones.items()), columns=['Timestamp', 'Average Tone'])
tone_df['Timestamp'] = pd.to_datetime(tone_df['Timestamp'], format='%Y%m%d%H%M%S')
tone_df


,Timestamp,Average Tone
0,2024-09-23 08:00:00,-0.531833
1,2024-09-23 08:15:00,-3.236250
2,2024-09-23 08:30:00,-0.025667
3,2024-09-23 08:45:00,1.060857
4,2024-09-23 09:00:00,-2.863167
...,...,...
59,2024-09-23 22:45:00,0.120400
60,2024-09-23 23:00:00,0.489000
61,2024-09-23 23:15:00,-0.892667
62,2024-09-23 23:30:00,1.537625


In [19]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tone_df)

https://docs.google.com/spreadsheets/d/1tgSrDpyAVm4Vosl544BMzFU83UgK8u_qWolMXrklrso/edit#gid=0


In [20]:
# Download intraday data using yfinance
nflx_intraday = yf.download(
    tickers="NFLX",
    start = "2024-09-23",
    end = "2024-09-23",
    interval = "15m",
    prepost = True, auto_adjust= False
)

# Save the data to a CSV file
nflx_intraday.to_csv("netflix_intraday_20240923.csv")

#Display DatFrame
nflx_intraday

[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NFLX']: YFPricesMissingError('possibly delisted; no price data found  (15m 2024-09-23 -> 2024-09-23) (Yahoo error = "15m data not available for startTime=1727064000 and endTime=1727064000. The requested range must be within the last 60 days.")')


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,NFLX,NFLX,NFLX,NFLX,NFLX,NFLX
Date,,,,,,


In [22]:
#importing files to google colab
from google.colab import files
uploaded = files.upload()

Saving WQU netflix_intraday_20240923.csv to WQU netflix_intraday_20240923.csv


In [23]:
# Open saved DataFrame - OPTIONAL
nflx_intraday = pd.read_csv('WQU netflix_intraday_20240923.csv')
nflx_intraday.set_index('Datetime', inplace=True)
nflx_intraday

,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2024-09-23 04:00:00,701.030000,701.030000,700.980000,700.980000,700.980000,0
2024-09-23 04:15:00,700.980000,700.980000,700.900000,700.960000,700.960000,0
2024-09-23 04:30:00,699.780000,699.780000,699.780000,699.780000,699.780000,0
2024-09-23 05:15:00,700.570000,700.990000,700.570000,700.790000,700.790000,0
2024-09-23 05:45:00,700.700000,701.500000,700.700000,701.500000,701.500000,0
2024-09-23 06:00:00,701.400000,701.500000,701.400000,701.500000,701.500000,0
2024-09-23 06:30:00,701.910000,701.910000,701.910000,701.910000,701.910000,0
2024-09-23 07:00:00,702.210000,702.210000,702.210000,702.210000,702.210000,0
2024-09-23 07:15:00,702.210000,702.210000,701.250000,701.250000,701.250000,0


In [24]:
# Convert Datetime to UTC timezone
nflx_intraday.index = pd.to_datetime(nflx_intraday.index).tz_localize('America/New_York').tz_convert('UTC')
nflx_intraday

,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2024-09-23 08:00:00+00:00,701.030000,701.030000,700.980000,700.980000,700.980000,0
2024-09-23 08:15:00+00:00,700.980000,700.980000,700.900000,700.960000,700.960000,0
2024-09-23 08:30:00+00:00,699.780000,699.780000,699.780000,699.780000,699.780000,0
2024-09-23 09:15:00+00:00,700.570000,700.990000,700.570000,700.790000,700.790000,0
2024-09-23 09:45:00+00:00,700.700000,701.500000,700.700000,701.500000,701.500000,0
2024-09-23 10:00:00+00:00,701.400000,701.500000,701.400000,701.500000,701.500000,0
2024-09-23 10:30:00+00:00,701.910000,701.910000,701.910000,701.910000,701.910000,0
2024-09-23 11:00:00+00:00,702.210000,702.210000,702.210000,702.210000,702.210000,0
2024-09-23 11:15:00+00:00,702.210000,702.210000,701.250000,701.250000,701.250000,0


In [26]:
# Create a candlestick trace
candelstick_trace = go.Candlestick (x =nflx_intraday.index,
                                    open = nflx_intraday['Open'],
                                    high = nflx_intraday['High'],
                                    low = nflx_intraday['Low'],
                                    close = nflx_intraday['Close'],
                                    name = 'Netflix Price')

#Create tone trace
tone_trace = go.Scatter(x =tone_df['Timestamp'],
                        y = tone_df['Average Tone'],
                        mode = 'lines',
                        name = 'Average Tone',
                        line = dict(color='blue', width =1),
                        yaxis ='y2')

#Create figure with both traces
fig = go.Figure(data=[candelstick_trace, tone_trace])

# Update Layout with secondary y-axis
fig.update_layout(title_text = 'Netflix Price and Average Tone',
                  yaxis_title = 'Price(USD)',
                  yaxis2 = dict(title = 'Average Tone',
                                overlaying = 'y',
                                side = 'right'))
fig.show()